<a href="https://colab.research.google.com/github/firatsayici/Breast-Cancer-Wisconsin-Analysis/blob/main/TR_GAN_Yakla%C5%9F%C4%B1m%C4%B1_ve_Film_G%C3%B6r%C3%BCnt%C3%BC_Kalitesi_Art%C4%B1rma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# -------------------------------------------------
# Utility Blocks
# -------------------------------------------------

class SpatialConv(nn.Module):
    """Spatial 2D Convolution"""
    def __init__(self, nf):
        super().__init__()
        self.conv = nn.Conv2d(nf, nf, 3, 1, 1)
        self.bn = nn.BatchNorm2d(nf)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class TemporalFusion(nn.Module):
    """
    Temporal residual fusion:
    F_t = Conv([F_t , F_t - F_{t-1}])
    """
    def __init__(self, nf):
        super().__init__()
        self.fusion = nn.Conv2d(nf * 2, nf, 1, 1, 0)

    def forward(self, curr_feat, prev_feat):
        diff = curr_feat - prev_feat
        combined = torch.cat([curr_feat, diff], dim=1)
        return self.fusion(combined)


# -------------------------------------------------
# (2+1)D Residual Block
# -------------------------------------------------

class ResidualBlock2plus1D(nn.Module):
    """
    Decomposed (2+1)D Residual Block
    Temporal Fusion → Spatial Conv → Residual Add
    """
    def __init__(self, nf, res_scale=0.1):
        super().__init__()
        self.temporal = TemporalFusion(nf)
        self.spatial = SpatialConv(nf)
        self.res_scale = res_scale

    def forward(self, x, prev_feat):
        identity = x
        out = self.temporal(x, prev_feat)
        out = self.spatial(out)
        return identity + out * self.res_scale


# -------------------------------------------------
# Texture Attention Module
# -------------------------------------------------

class TextureAttention(nn.Module):
    """
    Channel-aware texture attention
    (Lightweight SE-style modulation)
    """
    def __init__(self, nf, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(nf, nf // reduction, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(nf // reduction, nf, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn = self.fc(self.avg_pool(x))
        return x * attn


# -------------------------------------------------
# Upsampling Block (PixelShuffle)
# -------------------------------------------------

class UpsampleBlock(nn.Module):
    def __init__(self, nf, scale):
        super().__init__()
        layers = []
        for _ in range(int(scale // 2)):
            layers.append(nn.Conv2d(nf, nf * 4, 3, 1, 1))
            layers.append(nn.PixelShuffle(2))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.up = nn.Sequential(*layers)

    def forward(self, x):
        return self.up(x)


# -------------------------------------------------
# TR-GAN Generator
# -------------------------------------------------

class TR_GAN_Generator(nn.Module):
    """
    Temporal Residual GAN Generator
    Designed for Video Super-Resolution
    """

    def __init__(self,
                 in_nc=3,
                 out_nc=3,
                 nf=64,
                 nb=16,
                 upscale=4):

        super().__init__()

        self.nf = nf
        self.upscale = upscale

        # Initial Feature Extraction
        self.conv_first = nn.Conv2d(in_nc, nf, 3, 1, 1)

        # Residual Temporal Blocks
        self.res_blocks = nn.ModuleList(
            [ResidualBlock2plus1D(nf) for _ in range(nb)]
        )

        # Texture Attention
        self.texture_attention = TextureAttention(nf)

        # Feature Refinement
        self.conv_mid = nn.Conv2d(nf, nf, 3, 1, 1)

        # Upsampling
        self.upsample = UpsampleBlock(nf, upscale)

        # Output Layer
        self.conv_last = nn.Conv2d(nf, out_nc, 3, 1, 1)

    def forward(self, x_seq):
        """
        x_seq: (B, T, C, H, W)
        """
        B, T, C, H, W = x_seq.shape

        outputs = []
        prev_feat = torch.zeros(
            B, self.nf, H, W,
            device=x_seq.device
        )

        for t in range(T):

            x = x_seq[:, t]

            feat = self.conv_first(x)

            for block in self.res_blocks:
                feat = block(feat, prev_feat)

            feat = self.texture_attention(feat)

            feat = self.conv_mid(feat)

            out = self.upsample(feat)

            out = self.conv_last(out)

            outputs.append(out)

            prev_feat = feat.detach()

        return torch.stack(outputs, dim=1)

In [4]:
model = TR_GAN_Generator()
print("TR_GAN_Generator modeli başarıyla oluşturuldu.")

TR_GAN_Generator modeli başarıyla oluşturuldu.


In [5]:
# Örnek giriş tensörü oluşturalım
# Batch boyutu (B) = 1
# Zaman boyutu (T) = 5 (5 karelik bir video sekansı)
# Kanal sayısı (C) = 3 (RGB görüntüler)
# Giriş yüksekliği (H) = 64
# Giriş genişliği (W) = 64
dummy_input = torch.randn(1, 5, 3, 64, 64)

print(f"Giriş tensörünün şekli: {dummy_input.shape}")

# Modeli çalıştıralım
with torch.no_grad(): # Çıkarım yaparken gradyan hesaplamaya gerek yok
    output = model(dummy_input)

print(f"Çıkış tensörünün şekli: {output.shape}")

# Çıkışın beklenen süper çözünürlük ölçeği ile doğru boyutta olup olmadığını kontrol edelim
expected_h = dummy_input.shape[-2] * model.upscale
expected_w = dummy_input.shape[-1] * model.upscale
print(f"Beklenen çıkış yüksekliği: {expected_h}, Beklenen çıkış genişliği: {expected_w}")
assert output.shape[-2] == expected_h
assert output.shape[-1] == expected_w

print("Model başarıyla çalıştırıldı ve süper çözünürlüklü bir çıktı üretti!")

Giriş tensörünün şekli: torch.Size([1, 5, 3, 64, 64])
Çıkış tensörünün şekli: torch.Size([1, 5, 3, 256, 256])
Beklenen çıkış yüksekliği: 256, Beklenen çıkış genişliği: 256
Model başarıyla çalıştırıldı ve süper çözünürlüklü bir çıktı üretti!


Yukarıdaki kod, modelin beklediği formatta (5 karelik bir video sekansı) bir sahte giriş tensörü oluşturur ve bunu `TR_GAN_Generator` modeline besler.

Çıktı tensörünün şekli, giriş tensörünün zaman ve kanal boyutlarını korurken, yükseklik ve genişlik boyutlarının modelin `upscale` parametresi (bu durumda 4) kadar büyüdüğünü gösterir. Bu, modelin her bir giriş karesini süper çözünürlüğe çıkardığı ve birleştirilmiş süper çözünürlüklü video sekansını döndürdüğü anlamına gelir.